In [1]:
import hashlib, json, re, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, memcap, sysinfo

B = repo / "build/adapters"
git = lambda *a: subprocess.run(["git", *a], cwd=repo, capture_output=True, text=True).stdout.strip()
subprocess.run(["make", "adapters"], cwd=repo, check=True, capture_output=True)
assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
run_id = "memory-ceiling-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw = repo / "results/raw" / run_id
raw.mkdir(parents=True)
meta = {"run_id": run_id, "git_sha": git("rev-parse", "HEAD"), "dirty": bool(git("status", "--porcelain", "src", "Makefile")),
        "adapters": {b.name: hashlib.sha256(b.read_bytes()).hexdigest() for b in sorted(B.glob("*-adapter"))},
        "sysinfo": sysinfo.snapshot()}
(raw / "meta.json").write_text(json.dumps(meta, indent=1, default=str))
print({k: meta[k] for k in ("run_id", "git_sha", "dirty")})

# Notebook 07 capped each cell at 10 GiB. The spec calls a cell oom only above 16 GiB, the Mac's RAM, and
# macOS and the editor need about 2 GiB of that, so 14 GiB is the most a cell can have here.
CAP = 14 * memcap.GIB
LIMIT_S = 5400
BINARY = {"S0-go": "simplepir-adapter", "S0-cpp": "vsp-adapter", "S1": "vsp-adapter", "S2": "vsp-adapter",
          "M0": "apir-adapter", "M1": "apir-adapter", "S3": "apir-adapter"}

def swap_used_mib():
    m = re.search(r"used = ([\d.]+)M", subprocess.run(["sysctl", "vm.swapusage"], capture_output=True, text=True).stdout)
    return float(m.group(1))

def run_cell(arm, db, rb):
    """One cell, --phase all with one repetition, as in notebook 07. Returns a row; never raises for the cell's own failure."""
    name = f"{arm}-{db}-{rb}"
    out = raw / f"{name}.jsonl"
    argv = ["caffeinate", "-dims", str(B / BINARY[arm]), "--arm", arm, "--db-bytes", str(db), "--record-bytes", str(rb),
            "--phase", "all", "--reps", "1", "--warmup", "0", "--seed", "1234", "--fault", "none", "--out", str(out)]
    row = {"arm": arm, "db_bytes": db, "record_bytes": rb, "swap_before_mib": swap_used_mib()}
    with hygiene.Watch() as w:
        try:
            r = memcap.run(argv, timeout_s=LIMIT_S, cap_bytes=CAP)
            row.update(exit=r.returncode, over_cap=r.over_cap, timed_out=False, peak_gib=r.peak_footprint_bytes / memcap.GIB)
            (raw / f"{name}.stderr.log").write_text(r.stderr)
        except subprocess.TimeoutExpired:
            row.update(exit=None, over_cap=False, timed_out=True, peak_gib=None)
    row.update(wall_s=w.wall_s, hygiene=w.problems, swap_after_mib=swap_used_mib())
    recs = [json.loads(l) for l in out.read_text().splitlines()] if out.exists() else []
    row["statuses"] = sorted({r["status"] for r in recs})
    row["correct"] = next((r["correct"] for r in recs if r["phase"] == "online"), None)
    return row

{'run_id': 'memory-ceiling-20261111T083621Z', 'git_sha': '4e65b6a33acb4f60137b72d1da1fd327f42bfcc8', 'dirty': False}


In [2]:
# The cells notebook 07 stopped at its 10 GiB cap, fastest-growing first so S3 runs last.
MiB = 1 << 20
nb07 = repo / "results/raw/grid-feasibility-20261111T064956Z/feasibility.jsonl"
over = [json.loads(l) for l in nb07.read_text().splitlines()]
over = [(r["arm"], r["db_bytes"], r["record_bytes"]) for r in over if r["over_cap"]]
over.sort(key=lambda c: c[0] == "S3")
print(f"{len(over)} cells over notebook 07's 10 GiB cap")
rows = []
for arm, db, rb in over:
    r = run_cell(arm, db, rb)
    rows.append(r)
    state = ("OVER 14 GiB CAP" if r["over_cap"] else f"TIMED OUT (> {LIMIT_S // 60} min)" if r["timed_out"]
             else f"exit {r['exit']} {r['statuses']} correct={r['correct']}")
    print(f"{arm:6s} {db // MiB:5d} MiB {rb:5d} B  {state:34s} peak {r['peak_gib'] or 0:5.2f} GiB  wall {r['wall_s']:6.0f} s  "
          f"swap {r['swap_before_mib']:6.0f} -> {r['swap_after_mib']:6.0f} MiB  {r['hygiene'] or ''}", flush=True)
with open(raw / "ceiling.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

7 cells over notebook 07's 10 GiB cap


M1       128 MiB    32 B  exit 0 ['ok'] correct=True         peak 11.74 GiB  wall     14 s  swap   2632 ->   2701 MiB  


S1      1024 MiB    32 B  OVER 14 GiB CAP                    peak 14.00 GiB  wall    310 s  swap   2701 ->   2243 MiB  


S1      1024 MiB  8192 B  OVER 14 GiB CAP                    peak 14.00 GiB  wall    289 s  swap   2243 ->   2131 MiB  


S2      1024 MiB    32 B  OVER 14 GiB CAP                    peak 14.00 GiB  wall     47 s  swap   2131 ->   3054 MiB  


S2      1024 MiB  8192 B  OVER 14 GiB CAP                    peak 14.12 GiB  wall     19 s  swap   3054 ->   3022 MiB  


M1      1024 MiB    32 B  OVER 14 GiB CAP                    peak 14.22 GiB  wall     26 s  swap   3022 ->   3029 MiB  


S3      1024 MiB     0 B  OVER 14 GiB CAP                    peak 14.13 GiB  wall   1031 s  swap   3029 ->   3005 MiB  


In [3]:
fit = [r for r in rows if r["exit"] == 0 and r["statuses"] == ["ok"] and r["correct"] and not r["hygiene"]]
for label, sel in (("fit under 14 GiB", fit), ("need more than 14 GiB", [r for r in rows if r["over_cap"]]),
                   ("timed out", [r for r in rows if r["timed_out"]]),
                   ("other", [r for r in rows if r not in fit and not r["over_cap"] and not r["timed_out"]])):
    print(f"{label}: " + (", ".join(f"{r['arm']} {r['db_bytes'] // MiB} MiB {r['record_bytes']} B" for r in sel) or "none"))

fit under 14 GiB: M1 128 MiB 32 B
need more than 14 GiB: S1 1024 MiB 32 B, S1 1024 MiB 8192 B, S2 1024 MiB 32 B, S2 1024 MiB 8192 B, M1 1024 MiB 32 B, S3 1024 MiB 0 B
timed out: none
other: none
